In [25]:
import requests
import numpy as np
import json 
from pypdf import PdfReader

OLLAMA_URL = "http://localhost:11434"
EMBEDDING_MODEL = "nomic-embed-text"
GENERATION_MODEL = "llama3"

In [26]:
def extract_text(pdf_path):
    reader = PdfReader(pdf_path)
    return "\n".join(page.extract_text() for page in reader.pages)

text = extract_text("sample.pdf")

print(f"Extracted text {len(text)} characters")
#print text obeying \n 
print(text[:500])


Extracted text 2317 characters
The Lantern in the Rain
On the edge of a quiet village stood an old lighthouse that had not guided ships for decades.
People believed it had nothing left to offer except memories. A curious teenager named Arman,
however, visited it every weekend, fascinated by its weathered stones and endless view of the sea.
One stormy evening, while sheltering inside the tower, Arman discovered a dusty lantern hidden
beneath a wooden staircase. Unlike ordinary lanterns, it glowed warmly without oil or flame. A


In [27]:
def chunk_text(text, chunk_size=1000, overlap=200):
    chunks = []
    start = 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return chunks

chunks = chunk_text(text)
print(f"Created {len(chunks)} chunks")
print("First chunk: ", chunks[0][:10000])


Created 3 chunks
First chunk:  The Lantern in the Rain
On the edge of a quiet village stood an old lighthouse that had not guided ships for decades.
People believed it had nothing left to offer except memories. A curious teenager named Arman,
however, visited it every weekend, fascinated by its weathered stones and endless view of the sea.
One stormy evening, while sheltering inside the tower, Arman discovered a dusty lantern hidden
beneath a wooden staircase. Unlike ordinary lanterns, it glowed warmly without oil or flame. As he
lifted it, the wind outside calmed for a brief moment. An inscription read, 'Light shared is never lost.'
Intrigued, Arman carried it home.
Over the following weeks, strange but wonderful things happened. Whenever Arman used the
lantern to help someone—a lost traveler, an elderly neighbor, or frightened children during a
blackout—it shone brighter. Yet it dimmed whenever he tried to use it for personal gain. He soon
realized the lantern reflected kindness rath

In [28]:

def get_embedding(text):
    response = requests.post(
        f"{OLLAMA_URL}/api/embeddings",
        json={"model": EMBEDDING_MODEL, "prompt": text},
        timeout=60
    )
    response.raise_for_status()
    return response.json()["embedding"]

test_embedding = get_embedding("Hello, world!")
print(f"Embedding length: {len(test_embedding)}")

Embedding length: 768


In [29]:
index = []
for i, chunk in enumerate(chunks):
    embedding = get_embedding(chunk)
    index.append({"chunk": chunk, "embedding": embedding})
    print(f"Indexed chunk {i+1}/{len(chunks)}")

print("Done")


Indexed chunk 1/3
Indexed chunk 2/3
Indexed chunk 3/3
Done


In [30]:
print(len(chunks))


3


In [31]:
print("text length:", len(text))
print("chunks:", len(chunks))
print(chunks[:1] if chunks else "chunks is empty")

text length: 2317
chunks: 3
["The Lantern in the Rain\nOn the edge of a quiet village stood an old lighthouse that had not guided ships for decades.\nPeople believed it had nothing left to offer except memories. A curious teenager named Arman,\nhowever, visited it every weekend, fascinated by its weathered stones and endless view of the sea.\nOne stormy evening, while sheltering inside the tower, Arman discovered a dusty lantern hidden\nbeneath a wooden staircase. Unlike ordinary lanterns, it glowed warmly without oil or flame. As he\nlifted it, the wind outside calmed for a brief moment. An inscription read, 'Light shared is never lost.'\nIntrigued, Arman carried it home.\nOver the following weeks, strange but wonderful things happened. Whenever Arman used the\nlantern to help someone—a lost traveler, an elderly neighbor, or frightened children during a\nblackout—it shone brighter. Yet it dimmed whenever he tried to use it for personal gain. He soon\nrealized the lantern reflected kin

In [32]:
with open("index.json", "w") as f:
    json.dump(index, f)

In [33]:
def cosine_similarity(a,b):
    a,b = np.array(a), np.array(b)
    return np.dot(a,b)/(np.linalg.norm(a)*np.linalg.norm(b))
def retrieve_top_k(query, index, k=3):
    query_embedding = get_embedding(query)
    scored = [ {**item, "score": cosine_similarity(query_embedding, item["embedding"])} for item in index ]
    scored.sort(key=lambda x: x["score"], reverse=True)
    return scored[:k]

In [34]:
def ask_question(question, index, k=4):
    top_chunks = retrieve_top_k(question, index, k)
    context = "\n\n---\n\n".join(c["text"] for c in top_chunks)

    prompt = f"""Answer the question using only the context below. If the answer isn't in the context, say so.

Context:
{context}

Question: {question}"""

    response = client.models.generate_content(
        model="gemini-flash-latest",
        contents=prompt
    )
    return response.text

In [35]:
import google.genai as genai
import os
from dotenv import load_dotenv
load_dotenv(dotenv_path="D:/Projects/Ai Projects/PdfExtractingRagAgent/.env")  # adjust to your actual path
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))  # or load from env var, see note below
def ask_gemini(prompt):
    response = client.models.generate_content(
        model="gemini-flash-latest",
        contents=prompt
    )
    return response.text

answer = ask_gemini("What is the capital of France?")
print(answer)

The capital of France is **Paris**.


In [39]:
while True:
    q = input("Ask a question (or 'quit'): ")

    if q.lower() == "quit":
        break

    answer = ask_question(
        q,
        [{"text": item["chunk"], "embedding": item["embedding"]} for item in index]
    )

    print(f"\n{answer}\n")


Based on the provided context, here is a summary of the story:

Arman, a teenager fascinated by an old, abandoned lighthouse, finds a glowing dusty lantern hidden inside with the inscription, "Light shared is never lost." He discovers that the lantern shines brighter when he uses it to help others—such as a lost traveler or frightened children—and dims when he tries to use it for personal gain. 

Inspired by Arman's kindness, the villagers begin helping one another, transforming the village and turning the lighthouse into a lively community gathering place. Years later, during a severe storm that cuts out electricity, Arman takes the lantern up the lighthouse tower, where its brilliant light guides local fishing boats safely back to shore. 

Eventually, the lantern simply stops glowing. Arman is content because he realizes its purpose has been fulfilled: the true light of compassion now lives within the people of the village. As Arman grows old, children continue to climb the lighthou

ValueError: shapes (0,) and (768,) not aligned: 0 (dim 0) != 768 (dim 0)

In [37]:
import requests
res = requests.get(f"{OLLAMA_URL}/api/tags")
for m in res.json()["models"]:
    print(m["name"])

nomic-embed-text:latest
